In [ ]:
import pandas as pd
import numpy as np


In [ ]:
counties = gpd.read_file(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\2020 TIGER LINE COUNTY Files\tl_2025_us_county.shp"
)

counties["county_fips"] = counties["STATEFP"] + counties["COUNTYFP"]

county_lookup = counties[[
    "county_fips", "NAME", "STATEFP"
]].copy()

county_lookup.columns = ["county_fips", "county_name", "statefp"]

county_lookup["county_name"] = county_lookup["county_name"].str.lower().str.strip()


In [ ]:
county_lookup.shape


In [ ]:
import geopandas as gpd


In [ ]:
counties = gpd.read_file(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\2020 TIGER LINE COUNTY Files\tl_2025_us_county.shp"
)

counties["county_fips"] = counties["STATEFP"] + counties["COUNTYFP"]

county_lookup = counties[[
    "county_fips", "NAME", "STATEFP"
]].copy()

county_lookup.columns = ["county_fips", "county_name", "statefp"]

county_lookup["county_name"] = county_lookup["county_name"].str.lower().str.strip()


In [ ]:
county_lookup.shape


In [ ]:
county_lookup.head()


In [ ]:
# Step 2A: make merge keys clean + consistent
county_lookup = county_lookup.copy()

county_lookup["statefp"] = county_lookup["statefp"].astype(str).str.zfill(2)
county_lookup["county_fips"] = county_lookup["county_fips"].astype(str).str.zfill(5)
county_lookup["county_name"] = county_lookup["county_name"].astype(str).str.lower().str.strip()

county_lookup.head()


In [ ]:
pop_path = "co-est2024-pop.xlsx"  # OR r"C:\...\co-est2024-pop.xlsx"
pop_raw = pd.read_excel(pop_path)

pop_raw.columns
pop_raw.head()


In [ ]:
pop_path = "C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\US CENSUS 2020-2024 County Population\co-est2024-pop.xlsx"
pop_raw = pd.read_excel(pop_path)

pop_raw.columns
pop_raw.head()


In [ ]:
pop_path = C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\US CENSUS 2020-2024 County Population\co-est2024-pop.xlsx
pop_raw = pd.read_excel(pop_path)

pop_raw.columns
pop_raw.head()

In [ ]:
import os
os.getcwd()


In [ ]:
os.listdir()


In [ ]:
pop_path = r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\US CENSUS 2020-2024 County Population\co-est2024-pop.xlsx"
pop_raw = pd.read_excel(pop_path)
pop_raw.columns, pop_raw.head()


In [ ]:
os.listdir()

In [ ]:
# Rename population column for clarity
pop = pop_raw.rename(columns={"2024 Population": "population"}).copy()

# Remove leading dot and split "County, State"
pop["geo"] = pop["Geographic Area"].str.lstrip(".").str.strip()

pop["county_name"] = (
    pop["geo"]
    .str.replace(" County", "", regex=False)
    .str.split(",")
    .str[0]
    .str.lower()
    .str.strip()
)

pop["state_name"] = (
    pop["geo"]
    .str.split(",")
    .str[1]
    .str.lower()
    .str.strip()
)

pop = pop[["county_name", "state_name", "population"]]

pop.head()


In [ ]:
state_name_to_fips = {
    "alabama":"01","alaska":"02","arizona":"04","arkansas":"05","california":"06",
    "colorado":"08","connecticut":"09","delaware":"10","district of columbia":"11",
    "florida":"12","georgia":"13","hawaii":"15","idaho":"16","illinois":"17",
    "indiana":"18","iowa":"19","kansas":"20","kentucky":"21","louisiana":"22",
    "maine":"23","maryland":"24","massachusetts":"25","michigan":"26",
    "minnesota":"27","mississippi":"28","missouri":"29","montana":"30",
    "nebraska":"31","nevada":"32","new hampshire":"33","new jersey":"34",
    "new mexico":"35","new york":"36","north carolina":"37","north dakota":"38",
    "ohio":"39","oklahoma":"40","oregon":"41","pennsylvania":"42",
    "rhode island":"44","south carolina":"45","south dakota":"46","tennessee":"47",
    "texas":"48","utah":"49","vermont":"50","virginia":"51","washington":"53",
    "west virginia":"54","wisconsin":"55","wyoming":"56"
}

pop["STATEFP"] = pop["state_name"].map(state_name_to_fips)

pop["STATEFP"].isna().sum()


In [ ]:
county_lookup.head()


In [ ]:
county_full = pop.merge(
    county_lookup,
    left_on=["county_name", "STATEFP"],
    right_on=["county_name", "statefp"],
    how="left"
)

county_full[["county_name", "state_name", "population", "county_fips"]].head()


In [ ]:
print("Rows:", len(county_full))
print("Missing county_fips %:", county_full["county_fips"].isna().mean())
print("Unique county_fips:", county_full["county_fips"].nunique())
print("Population sum:", county_full["population"].sum())


In [ ]:
import re

def norm_county_name(s: str) -> str:
    s = str(s).lower().strip()
    s = s.lstrip(".").strip()

    # Remove common county-equivalent suffixes (order matters: longer phrases first)
    suffixes = [
        " city and borough",
        " census area",
        " municipality",
        " parish",
        " borough",
        " city",      # for independent cities (e.g., "baltimore city")
        " county",
    ]
    for suf in suffixes:
        if s.endswith(suf):
            s = s[: -len(suf)].strip()

    # normalize punctuation/spaces
    s = re.sub(r"\s+", " ", s)
    return s

# --- Rebuild pop cleaned ---
pop = pop_raw.rename(columns={"2024 Population": "population"}).copy()
pop["geo"] = pop["Geographic Area"].astype(str).str.lstrip(".").str.strip()

pop["county_name"] = pop["geo"].str.split(",").str[0].apply(norm_county_name)
pop["state_name"]  = pop["geo"].str.split(",").str[1].str.lower().str.strip()

pop["STATEFP"] = pop["state_name"].map(state_name_to_fips)

pop = pop[["county_name", "state_name", "STATEFP", "population"]]

# --- Normalize county_lookup too (VERY important) ---
county_lookup2 = county_lookup.copy()
county_lookup2["county_name"] = county_lookup2["county_name"].apply(norm_county_name)

# --- Merge again ---
county_full = pop.merge(
    county_lookup2[["county_fips", "county_name", "statefp"]],
    left_on=["county_name", "STATEFP"],
    right_on=["county_name", "statefp"],
    how="left"
)

print("Rows:", len(county_full))
print("Missing county_fips %:", county_full["county_fips"].isna().mean())
print("Unique county_fips:", county_full["county_fips"].nunique())
print("Population sum:", county_full["population"].sum())


In [ ]:
county_derms.columns


In [ ]:
county_derms = pd.read_csv("zip_dermatologist_counts.csv", dtype={"county_fips": str})


In [ ]:
county_derms = pd.read_csv(
    r"C:\All Programs\Med School\Appalachia Project Extended\...\zip_dermatologist_counts.csv",
    dtype={"county_fips": str}
)


In [ ]:
county_derms = pd.read_csv(
    r"C:\Users\victo\zip_dermatologist_counts.csv",
    dtype={"county_fips": str}
)



In [ ]:
county_derms.head()
county_derms.shape
county_derms["county_fips"].nunique()
county_derms["dermatologist_count"].sum()


In [ ]:
import pandas as pd

county_derms_path = r"C:\Users\victo\zip_dermatologist_counts.csv"  # <-- your file path
county_derms = pd.read_csv(county_derms_path)

print(county_derms.shape)
print(county_derms.columns.tolist())
county_derms.head()


In [ ]:
county_derms = county_derms.rename(columns={
    "Provider Business Practice Location Address Postal Code": "zip"
})

county_derms["zip"] = county_derms["zip"].astype(str).str.zfill(5)

county_derms.head()


In [ ]:
county_derms["zip"].nunique()
county_derms["dermatologist_count"].sum()


In [ ]:
crosswalk = pd.read_csv(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\zip_to_county_crosswalk.csv",
    dtype={"ZIP": str, "COUNTY": str}
)

crosswalk["ZIP"] = crosswalk["ZIP"].str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].str.zfill(5)

crosswalk.head()


In [ ]:
crosswalk = pd.read_csv(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\HUD_USPS_ZIP_COUNTY_092025.xlsx",
    dtype={"ZIP": str, "COUNTY": str}
)

crosswalk["ZIP"] = crosswalk["ZIP"].str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].str.zfill(5)

crosswalk.head()


In [ ]:
crosswalk = pd.read_csv(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File",
    dtype={"ZIP": str, "COUNTY": str}
)

crosswalk["ZIP"] = crosswalk["ZIP"].str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].str.zfill(5)

crosswalk.head()


In [ ]:
crosswalk = pd.read_csv(
    r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\HUD_USPS_ZIP_COUNTY_092025.xlsx",
    dtype={"ZIP": str, "COUNTY": str}
)

crosswalk["ZIP"] = crosswalk["ZIP"].str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].str.zfill(5)

crosswalk.head()


In [ ]:
import pandas as pd

cw_path = r"C:\All Programs\Med School\Appalachia Project Extended\NPI registry Data\Large Data File\HUD_USPS_ZIP_COUNTY_092025.xlsx"

crosswalk = pd.read_excel(cw_path, dtype={"ZIP": str, "COUNTY": str})

# Standardize formats
crosswalk["ZIP"] = crosswalk["ZIP"].str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].str.zfill(5)

crosswalk.head()


In [ ]:
print(crosswalk.columns)
print(crosswalk[["ZIP","COUNTY"]].head())


In [ ]:
zip_derms = county_derms.rename(
    columns={"Provider Business Practice Location Address Postal Code": "ZIP"}
).copy()

zip_derms["ZIP"] = zip_derms["ZIP"].astype(str).str.zfill(5)

print(zip_derms.head())
print("ZIPs:", zip_derms["ZIP"].nunique())
print("Total derms:", zip_derms["dermatologist_count"].sum())


In [ ]:
zip_derms = county_derms.rename(
    columns={"Provider Business Practice Location Address Postal Code": "ZIP"}
).copy()


In [ ]:
zip_derms = pd.read_csv(
    r"C:\Users\victo\zip_dermatologist_counts.csv"
)

print(zip_derms.columns)
zip_derms.head()


In [ ]:
zip_derms = zip_derms.rename(
    columns={"Provider Business Practice Location Address Postal Code": "ZIP"}
)

zip_derms["ZIP"] = zip_derms["ZIP"].astype(str).str.zfill(5)

print(zip_derms.head())
print("ZIPs:", zip_derms["ZIP"].nunique())
print("Total derms:", zip_derms["dermatologist_count"].sum())


In [ ]:
zip_alloc = zip_derms.merge(
    crosswalk[["ZIP", "COUNTY", "TOT_RATIO"]],
    on="ZIP",
    how="left"
)


In [ ]:
zip_derms.head()
zip_derms.columns


In [ ]:
zip_derms["ZIP"] = zip_derms["ZIP"].astype(str).str.zfill(5)

print(zip_derms.head())
print("ZIPs:", zip_derms["ZIP"].nunique())
print("Total derms:", zip_derms["dermatologist_count"].sum())


In [ ]:
crosswalk["ZIP"] = crosswalk["ZIP"].astype(str).str.zfill(5)
crosswalk["COUNTY"] = crosswalk["COUNTY"].astype(str).str.zfill(5)

print(crosswalk[["ZIP","COUNTY","TOT_RATIO"]].head())


In [ ]:
zip_alloc = zip_derms.merge(
    crosswalk[["ZIP", "COUNTY", "TOT_RATIO"]],
    on="ZIP",
    how="left"
)

print("Rows:", len(zip_alloc))
print("Unmatched ZIP %:", zip_alloc["COUNTY"].isna().mean())
zip_alloc.head()


In [ ]:
zip_alloc["derms_alloc"] = zip_alloc["dermatologist_count"] * zip_alloc["TOT_RATIO"]

county_derms = (
    zip_alloc.dropna(subset=["COUNTY"])
    .groupby("COUNTY", as_index=False)["derms_alloc"]
    .sum()
    .rename(columns={"COUNTY": "county_fips", "derms_alloc": "dermatologist_count"})
)

print("Counties:", county_derms["county_fips"].nunique())
print("Allocated total derms:", county_derms["dermatologist_count"].sum())
county_derms.head()


In [ ]:
print("Crosswalk unique counties:", crosswalk["COUNTY"].astype(str).str.zfill(5).nunique())
print("Crosswalk states (sample):", crosswalk["USPS_ZIP_PREF_STATE"].value_counts().head(15))


In [ ]:
print("ZIP derm rows:", len(zip_derms))
print("Unique derm ZIPs:", zip_derms["ZIP"].nunique())


In [ ]:
county_derms["county_fips"] = county_derms["county_fips"].astype(str).str.zfill(5)


In [ ]:
counties["county_fips"] = (counties["STATEFP"].astype(str).str.zfill(2) +
                           counties["COUNTYFP"].astype(str).str.zfill(3))

county_master = counties[["county_fips"]].drop_duplicates().copy()
print("Total counties in master:", county_master["county_fips"].nunique())


In [ ]:
county_full = county_master.merge(county_derms, on="county_fips", how="left")
county_full["dermatologist_count"] = county_full["dermatologist_count"].fillna(0)

print("Counties total:", county_full["county_fips"].nunique())
print("Counties with derms >0:", (county_full["dermatologist_count"] > 0).sum())
print("Total derms (allocated):", county_full["dermatologist_count"].sum())


In [ ]:
print("Total derms in ZIP file:", zip_derms["dermatologist_count"].sum())


In [ ]:
# How many derms got stuck in unmatched ZIPs?
unmatched_rows = zip_alloc["COUNTY"].isna()
unmatched_derms = zip_alloc.loc[unmatched_rows, "dermatologist_count"].sum()

print("Unmatched derms (not allocated):", unmatched_derms)
print("Unmatched ZIP rows:", unmatched_rows.sum())
print("Unmatched unique ZIPs:", zip_alloc.loc[unmatched_rows, "ZIP"].nunique())

# Show the worst offenders (ZIPs with most derms but no county match)
(zip_alloc.loc[unmatched_rows, ["ZIP", "dermatologist_count"]]
 .drop_duplicates()
 .sort_values("dermatologist_count", ascending=False)
 .head(20))


In [ ]:
# Keep only matched ZIP->county rows
za = zip_alloc.dropna(subset=["COUNTY"]).copy()

# Ensure numeric
za["TOT_RATIO"] = pd.to_numeric(za["TOT_RATIO"], errors="coerce").fillna(0)

# Normalize TOT_RATIO within each ZIP (so each ZIP's ratios sum to 1)
ratio_sum = za.groupby("ZIP")["TOT_RATIO"].transform("sum")
za = za[ratio_sum > 0].copy()
za["ratio_norm"] = za["TOT_RATIO"] / ratio_sum

# Allocate derms
za["derms_alloc"] = za["dermatologist_count"] * za["ratio_norm"]

county_derms = (
    za.groupby("COUNTY", as_index=False)["derms_alloc"].sum()
      .rename(columns={"COUNTY": "county_fips", "derms_alloc": "dermatologist_count"})
)

county_derms["county_fips"] = county_derms["county_fips"].astype(str).str.zfill(5)

print("Allocated total derms (matched ZIPs):", county_derms["dermatologist_count"].sum())
print("Still-unmatched derms:", unmatched_derms)
print("ZIP total derms:", zip_derms["dermatologist_count"].sum())


In [ ]:
county_full = county_master.merge(county_derms, on="county_fips", how="left")
county_full["dermatologist_count"] = county_full["dermatologist_count"].fillna(0)

print("Counties total:", county_full["county_fips"].nunique())
print("Total derms (allocated):", county_full["dermatologist_count"].sum())


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total counties",
        "Total population",
        "Total dermatologists (allocated)",
        "Appalachia population",
        "Appalachia dermatologists",
        "Dermatologists per 100k (Appalachia)",
        "Dermatologists per 100k (US)",
        "Rate ratio (Appalachia / US)",
        "Unmatched dermatologists (ZIP-level)"
    ],
    "Value": [
        county_full["county_fips"].nunique(),
        county_full["population"].sum(),
        county_full["dermatologist_count"].sum(),
        county_full.loc[county_full["is_appalachia"], "population"].sum(),
        county_full.loc[county_full["is_appalachia"], "dermatologist_count"].sum(),
        (county_full.loc[county_full["is_appalachia"], "dermatologist_count"].sum()
         / county_full.loc[county_full["is_appalachia"], "population"].sum()) * 100_000,
        (county_full["dermatologist_count"].sum()
         / county_full["population"].sum()) * 100_000,
        ((county_full.loc[county_full["is_appalachia"], "dermatologist_count"].sum()
          / county_full.loc[county_full["is_appalachia"], "population"].sum())
         /
         (county_full["dermatologist_count"].sum()
          / county_full["population"].sum())),
        34
    ]
})

summary


In [ ]:
county_full.columns


In [ ]:
[c for c in county_full.columns if "pop" in c.lower()]


In [ ]:
[name for name in globals() if "pop" in name.lower()]


In [ ]:
pop.columns
pop.head()


In [ ]:
county_full = county_full.merge(
    pop[["county_fips", "population"]],
    on="county_fips",
    how="left"
)


In [ ]:
pop.columns
pop.head()


In [ ]:
import pandas as pd
import numpy as np

# 1) Ensure consistent types/format
county_lookup = county_lookup.copy()
county_lookup["statefp"] = county_lookup["statefp"].astype(str).str.zfill(2)
county_lookup["county_name"] = county_lookup["county_name"].astype(str).str.lower().str.strip()

pop = pop.copy()
pop["STATEFP"] = pop["STATEFP"].astype(str).str.zfill(2)
pop["county_name"] = pop["county_name"].astype(str).str.lower().str.strip()

# 2) Merge pop -> county_lookup to get county_fips
pop_fips = pop.merge(
    county_lookup[["STATEFP" if "STATEFP" in county_lookup.columns else "statefp", "county_name", "county_fips"]]
    .rename(columns={"statefp": "STATEFP"}),   # ensures the join key is STATEFP
    on=["STATEFP", "county_name"],
    how="left"
)

print("Rows pop:", len(pop_fips))
print("Missing county_fips %:", pop_fips["county_fips"].isna().mean())
print("Population sum (pop):", pop_fips["population"].sum())

# 3) Collapse to one row per county_fips (safe if duplicates exist)
pop_by_county = (pop_fips.dropna(subset=["county_fips"])
                 .groupby("county_fips", as_index=False)["population"].sum())

print("Unique counties w/ population:", pop_by_county["county_fips"].nunique())


In [ ]:
county_full = county_full.copy()
county_full["county_fips"] = county_full["county_fips"].astype(str).str.zfill(5)

county_full = county_full.merge(pop_by_county, on="county_fips", how="left")

print("Missing population %:", county_full["population"].isna().mean())
print("Total population (matched counties):", county_full["population"].sum())

county_full["derms_per_100k"] = (county_full["dermatologist_count"] / county_full["population"]) * 100_000


In [ ]:
# arc should already exist and contain county_fips (you showed arc.columns = ['county_fips'])
arc_fips = set(arc["county_fips"].astype(str).str.zfill(5))
county_full["is_appalachia"] = county_full["county_fips"].isin(arc_fips)

app = county_full[county_full["is_appalachia"]].copy()

us_rate  = county_full["dermatologist_count"].sum() / county_full["population"].sum() * 100_000
app_rate = app["dermatologist_count"].sum() / app["population"].sum() * 100_000

summary = pd.DataFrame({
    "Metric": [
        "Total counties (in county_full)",
        "Counties with population",
        "Total population (matched counties)",
        "Total dermatologists (allocated, matched ZIPs)",
        "Appalachia population",
        "Appalachia dermatologists (allocated)",
        "Derms per 100k (US)",
        "Derms per 100k (Appalachia)",
        "Rate ratio (Appalachia / US)",
        "Unmatched dermatologists (ZIP-level)"
    ],
    "Value": [
        county_full["county_fips"].nunique(),
        county_full["population"].notna().sum(),
        county_full["population"].sum(),
        county_full["dermatologist_count"].sum(),
        app["population"].sum(),
        app["dermatologist_count"].sum(),
        us_rate,
        app_rate,
        app_rate / us_rate,
        zip_derms["dermatologist_count"].sum() - county_full["dermatologist_count"].sum()
    ]
})

summary


In [ ]:
import pandas as pd

# 1) Load ARC counties (adjust path if needed)
arc_path = r"C:\All Programs\Med School\Appalachia Project Extended\Appalachian Counties Served By ARC\Appalachian-Counties-Served-by-ARC_2021.xlsx"
arc_raw = pd.read_excel(arc_path)

print("ARC columns:", arc_raw.columns.tolist())
arc_raw.head()


In [ ]:
# 2) Auto-detect the FIPS column in ARC file
fips_col = None
for c in arc_raw.columns:
    if str(c).strip().lower() in {"fips", "county fips", "county_fips", "geoid"}:
        fips_col = c
        break

if fips_col is None:
    # fallback: pick any column containing 'fips'
    candidates = [c for c in arc_raw.columns if "fips" in str(c).lower()]
    print("FIPS candidates:", candidates)
    fips_col = candidates[0]  # if this errors, tell me candidates list

arc = arc_raw.copy()
arc["county_fips"] = arc[fips_col].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)

print("ARC rows:", len(arc))
print("Unique ARC county_fips:", arc["county_fips"].nunique())
arc[["county_fips"]].head()


In [ ]:
arc_fips = set(arc["county_fips"].astype(str).str.zfill(5))
county_full["is_appalachia"] = county_full["county_fips"].isin(arc_fips)

app = county_full[county_full["is_appalachia"]].copy()

us_rate  = county_full["dermatologist_count"].sum() / county_full["population"].sum() * 100_000
app_rate = app["dermatologist_count"].sum() / app["population"].sum() * 100_000

summary = pd.DataFrame({
    "Metric": [
        "Total counties (in county_full)",
        "Counties with population",
        "Total population (matched counties)",
        "Total dermatologists (allocated, matched ZIPs)",
        "Appalachia population",
        "Appalachia dermatologists (allocated)",
        "Derms per 100k (US)",
        "Derms per 100k (Appalachia)",
        "Rate ratio (Appalachia / US)",
        "Unmatched dermatologists (ZIP-level)"
    ],
    "Value": [
        county_full["county_fips"].nunique(),
        county_full["population"].notna().sum(),
        county_full["population"].sum(),
        county_full["dermatologist_count"].sum(),
        app["population"].sum(),
        app["dermatologist_count"].sum(),
        us_rate,
        app_rate,
        app_rate / us_rate,
        zip_derms["dermatologist_count"].sum() - county_full["dermatologist_count"].sum()
    ]
})

summary


In [ ]:
output_path = r"C:\Users\victo\AppalachiaDerm_results.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    county_full.to_excel(writer, sheet_name="County_Level_Data", index=False)
    summary.to_excel(writer, sheet_name="Summary_Stats", index=False)

print("Saved to:", output_path)


In [ ]:
output_path = r"C:\Users\victo\AppalachiaDerm_results.xlsx"

with pd.ExcelWriter(output_path) as writer:
    county_full.to_excel(writer, sheet_name="County_Level_Data", index=False)
    summary.to_excel(writer, sheet_name="Summary_Stats", index=False)

print("Saved to:", output_path)


In [ ]:
# Totals
app_derms = app["dermatologist_count"].sum()
app_pop   = app["population"].sum()

us_derms  = county_full["dermatologist_count"].sum()
us_pop    = county_full["population"].sum()

# Rates per person (not per 100k yet)
rate_app = app_derms / app_pop
rate_us  = us_derms / us_pop


In [ ]:
rate_ratio = rate_app / rate_us
rate_ratio


In [ ]:
import numpy as np

# Standard error of log(RR)
se_log_rr = np.sqrt(1/app_derms + 1/us_derms)

# 95% CI
ci_lower = np.exp(np.log(rate_ratio) - 1.96 * se_log_rr)
ci_upper = np.exp(np.log(rate_ratio) + 1.96 * se_log_rr)

ci_lower, ci_upper


In [ ]:
from scipy.stats import norm

z = np.log(rate_ratio) / se_log_rr
p_value = 2 * (1 - norm.cdf(abs(z)))

z, p_value


In [ ]:
stats_summary = pd.DataFrame({
    "Metric": [
        "Dermatologists per 100k (US)",
        "Dermatologists per 100k (Appalachia)",
        "Rate Ratio (Appalachia / US)",
        "95% CI Lower",
        "95% CI Upper",
        "p-value"
    ],
    "Value": [
        rate_us * 100_000,
        rate_app * 100_000,
        rate_ratio,
        ci_lower,
        ci_upper,
        p_value
    ]
})

stats_summary


In [ ]:
stats_summary


In [ ]:
output_path = r"C:\Users\victo\AppalachiaDerm_results.xlsx"

with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"   # change to "new" if you want versioned sheets
) as writer:
    
    stats_summary.to_excel(
        writer,
        sheet_name="Rate_Comparison_Stats",
        index=False
    )

print("Saved statistical results to Excel.")


In [ ]:
with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    county_full.to_excel(
        writer,
        sheet_name="County_Level_Data",
        index=False
    )


In [ ]:
with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    county_full.to_excel(
        writer,
        sheet_name="County_Level_Data",
        index=False
    )


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---- Paths ----
fig_dir = r"C:\Users\victo\AppalachiaDerm\figures"
os.makedirs(fig_dir, exist_ok=True)

png_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.png")
pdf_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.pdf")

# ---- Pull numbers from stats_summary (robust) ----
# Expect stats_summary has columns: ["Metric","Value"] and these Metric labels:
# "Derms per 100k (US)", "Derms per 100k (Appalachia)", "Rate ratio (Appalachia / US)",
# "RR 95% CI lower", "RR 95% CI upper", "p-value"
def get_val(metric_name: str) -> float:
    s = stats_summary.loc[stats_summary["Metric"].eq(metric_name), "Value"]
    if s.empty:
        raise KeyError(f"Couldn't find Metric='{metric_name}' in stats_summary")
    return float(s.iloc[0])

us_rate  = get_val("Derms per 100k (US)")
app_rate = get_val("Derms per 100k (Appalachia)")
rr       = get_val("Rate ratio (Appalachia / US)")
ci_lo    = get_val("RR 95% CI lower")
ci_hi    = get_val("RR 95% CI upper")
pval     = get_val("p-value")

# Optional: counts/pops if you want Poisson error bars on the rate bars
# If these are present in your earlier summary table, uncomment and use:
# us_pop  = get_val("Total population (matched counties)")
# us_derm = get_val("Total dermatologists (allocated, matched ZIPs)")
# app_pop = get_val("Appalachia population")
# app_derm= get_val("Appalachia dermatologists (allocated)")
# us_se   = (np.sqrt(us_derm) / us_pop) * 100_000
# app_se  = (np.sqrt(app_derm) / app_pop) * 100_000

# ---- Plot ----
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12
})

fig = plt.figure(figsize=(8.5, 4.8), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.35)

# Panel A: rates
ax1 = fig.add_subplot(gs[0, 0])
groups = ["US (matched counties)", "Appalachia (ARC)"]
rates = [us_rate, app_rate]

ax1.bar(groups, rates)
ax1.set_ylabel("Dermatologists per 100,000")
ax1.set_title("A. Per-capita dermatologist density")
ax1.set_ylim(0, max(rates) * 1.25)

# Value labels
for i, v in enumerate(rates):
    ax1.text(i, v + max(rates)*0.03, f"{v:.2f}", ha="center", va="bottom")

ax1.tick_params(axis="x", rotation=18)

# Panel B: RR forest-style
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("B. Rate ratio (Appalachia / US)")
ax2.set_xlabel("Rate ratio (RR)")
ax2.set_yticks([0])
ax2.set_yticklabels(["Appalachia vs US"])
ax2.axvline(1.0, linestyle="--")  # null

# CI line + point
ax2.plot([ci_lo, ci_hi], [0, 0], linewidth=2)
ax2.plot([rr], [0], marker="o")

# Limits with padding
xmin = max(0, min(ci_lo, 1.0) * 0.85)
xmax = max(ci_hi, 1.0) * 1.15
ax2.set_xlim(xmin, xmax)
ax2.set_ylim(-0.8, 0.8)

# Annotation
ax2.text(
    0.02, 0.95,
    f"RR = {rr:.3f}\n95% CI: {ci_lo:.3f}–{ci_hi:.3f}\np = {pval:.3g}",
    transform=ax2.transAxes,
    ha="left", va="top"
)

fig.suptitle("Dermatologist access: Appalachia vs United States (per-capita)", y=1.02)
fig.tight_layout()

# Save
fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()

print("Saved:")
print("PNG:", png_path)
print("PDF:", pdf_path)


In [ ]:
# If your table is named `summary` (from earlier), use it:
stats_summary = summary.copy()

# sanity check
stats_summary["Metric"].tolist()[:15]


In [ ]:
import re
import numpy as np

def get_val_fuzzy(needles):
    """Return Value where Metric contains ALL substrings in `needles` (case-insensitive)."""
    m = stats_summary["Metric"].astype(str)
    mask = np.ones(len(stats_summary), dtype=bool)
    for n in needles:
        mask &= m.str.contains(re.escape(n), case=False, na=False)
    hits = stats_summary.loc[mask, ["Metric", "Value"]]
    if hits.empty:
        raise KeyError(f"No Metric contains: {needles}\nAvailable metrics:\n{stats_summary['Metric'].tolist()}")
    if len(hits) > 1:
        print("Multiple matches found, using first:\n", hits)
    return float(hits["Value"].iloc[0])

# These should work even if your labels say "Derms per 100k (U.S.)" etc.
us_rate  = get_val_fuzzy(["derms per 100k", "us"])
app_rate = get_val_fuzzy(["derms per 100k", "appalach"])
rr       = get_val_fuzzy(["rate ratio", "appalach"])

print(us_rate, app_rate, rr)


In [ ]:
import os
import matplotlib.pyplot as plt

fig_dir = r"C:\Users\victo\AppalachiaDerm\figures"
os.makedirs(fig_dir, exist_ok=True)

png_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.png")
pdf_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.pdf")

plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 12})

fig = plt.figure(figsize=(8.5, 4.8), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.35)

# Panel A
ax1 = fig.add_subplot(gs[0, 0])
groups = ["US (matched counties)", "Appalachia (ARC)"]
rates = [us_rate, app_rate]
ax1.bar(groups, rates)
ax1.set_ylabel("Dermatologists per 100,000")
ax1.set_title("A. Per-capita dermatologist density")
ax1.set_ylim(0, max(rates) * 1.25)
for i, v in enumerate(rates):
    ax1.text(i, v + max(rates)*0.03, f"{v:.2f}", ha="center", va="bottom")
ax1.tick_params(axis="x", rotation=18)

# Panel B (RR only)
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("B. Rate ratio (Appalachia / US)")
ax2.set_xlabel("Rate ratio (RR)")
ax2.set_yticks([0])
ax2.set_yticklabels(["Appalachia vs US"])
ax2.axvline(1.0, linestyle="--")
ax2.plot([rr], [0], marker="o")
ax2.set_ylim(-0.8, 0.8)
ax2.set_xlim(0, max(1.2, rr * 1.3))
ax2.text(0.02, 0.95, f"RR = {rr:.3f}", transform=ax2.transAxes, ha="left", va="top")

fig.suptitle("Dermatologist access: Appalachia vs United States (per-capita)", y=1.02)
fig.tight_layout()

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()

print("Saved:")
print("PNG:", png_path)
print("PDF:", pdf_path)


In [ ]:
import os
import matplotlib.pyplot as plt

fig_dir = r"C:\Users\victo\AppalachiaDerm\figures"
os.makedirs(fig_dir, exist_ok=True)

png_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.png")
pdf_path = os.path.join(fig_dir, "derm_rate_comparison_US_vs_Appalachia.pdf")

plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 12})

fig = plt.figure(figsize=(8.5, 4.8), dpi=300, constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0])

# Panel A
ax1 = fig.add_subplot(gs[0, 0])
groups = ["US (matched counties)", "Appalachia (ARC)"]
rates = [us_rate, app_rate]
ax1.bar(groups, rates)
ax1.set_ylabel("Dermatologists per 100,000")
ax1.set_title("A. Per-capita dermatologist density")
ax1.set_ylim(0, max(rates) * 1.25)
for i, v in enumerate(rates):
    ax1.text(i, v + max(rates)*0.03, f"{v:.2f}", ha="center", va="bottom")
ax1.tick_params(axis="x", rotation=12)


# Panel B (RR only)
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("B. Rate ratio (Appalachia / US)")
ax2.set_xlabel("Rate ratio (RR)")
ax2.set_yticks([])
ax2.text(-0.02, 0, "Appalachia vs US", ha="right", va="center", transform=ax2.transAxes)
ax2.axvline(1.0, linestyle="--")
ax2.plot([rr], [0], marker="o")
ax2.set_ylim(-0.8, 0.8)
ax2.set_xlim(0, max(1.2, rr * 1.3))
ax2.text(0.02, 0.95, f"RR = {rr:.3f}", transform=ax2.transAxes, ha="left", va="top")

fig.suptitle("Dermatologist access: Appalachia vs United States (per-capita)", y=1.02)

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()

print("Saved:")
print("PNG:", png_path)
print("PDF:", pdf_path)


In [ ]:
import numpy as np
from scipy.stats import poisson
from statsmodels.stats.weightstats import ztest


In [ ]:
# Totals
us_derms = county_full["dermatologist_count"].sum()
us_pop   = county_full["population"].sum()

app_derms = app["dermatologist_count"].sum()
app_pop   = app["population"].sum()


In [ ]:
us_rate  = us_derms  / us_pop  * 100_000
app_rate = app_derms / app_pop * 100_000


In [ ]:
def poisson_ci(count, exposure, alpha=0.05):
    lower = poisson.ppf(alpha/2, count) / exposure * 100_000
    upper = poisson.ppf(1 - alpha/2, count) / exposure * 100_000
    return lower, upper

us_ci  = poisson_ci(us_derms, us_pop)
app_ci = poisson_ci(app_derms, app_pop)


In [ ]:
rr = app_rate / us_rate

se_log_rr = np.sqrt(1/app_derms + 1/us_derms)
ci_rr = (
    np.exp(np.log(rr) - 1.96 * se_log_rr),
    np.exp(np.log(rr) + 1.96 * se_log_rr),
)

z = np.log(rr) / se_log_rr
p_value = 2 * (1 - poisson.cdf(abs(z), mu=0))


In [ ]:
import matplotlib.pyplot as plt

labels = ["US (matched counties)", "Appalachia (ARC)"]
rates  = [us_rate, app_rate]
cis    = [us_ci, app_ci]

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

y = np.arange(len(labels))

ax.errorbar(
    rates, y,
    xerr=[
        [rates[i] - cis[i][0] for i in range(2)],
        [cis[i][1] - rates[i] for i in range(2)]
    ],
    fmt="o",
    color="#1f77b4",
    capsize=4
)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("Dermatologists per 100,000 population")
ax.set_title("Dermatologist density with 95% confidence intervals")

for i, r in enumerate(rates):
    ax.text(r + 0.15, i, f"{r:.2f}", va="center")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ["US (matched counties)", "Appalachia (ARC)"]
rates = [us_rate, app_rate]
cis = [us_ci, app_ci]

# CI widths
lower_err = [rates[i] - cis[i][0] for i in range(2)]
upper_err = [cis[i][1] - rates[i] for i in range(2)]

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

bars = ax.bar(
    x,
    rates,
    yerr=[lower_err, upper_err],
    capsize=6,
    color=["#1f77b4", "#ff7f0e"],
    edgecolor="black",
    linewidth=0.6
)

# Labels and formatting
ax.set_ylabel("Dermatologists per 100,000 population")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, max(rates) + 1.5)

ax.set_title("Dermatologist density with 95% confidence intervals")

# Annotate values on bars
for i, v in enumerate(rates):
    ax.text(i, v + 0.15, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- INPUTS (use your existing variables if you already have them) ---
us_rate  = float(us_rate)
app_rate = float(app_rate)

us_ci  = tuple(us_ci)   # (low, high)
app_ci = tuple(app_ci)  # (low, high)

# For a p-value, we need counts & population (Poisson rate comparison)
# If you have them in your summary already, plug them here:
us_derms  = float(us_derms)   # total derms in matched counties
us_pop    = float(us_pop)     # total population in matched counties
app_derms = float(app_derms)  # total derms in ARC counties
app_pop   = float(app_pop)    # total population in ARC counties

# --- Compute RR and two-sided p-value for difference in Poisson rates ---
# rate per person
r1 = app_derms / app_pop
r0 = us_derms / us_pop

rr = r1 / r0
se_log_rr = np.sqrt(1/app_derms + 1/us_derms)
z = np.log(rr) / se_log_rr
p_value = 2 * (1 - norm.cdf(abs(z)))

rr_ci_low = np.exp(np.log(rr) - 1.96 * se_log_rr)
rr_ci_high = np.exp(np.log(rr) + 1.96 * se_log_rr)

# Convert RR to your "per 100k" rate ratio matches already
# (it’s the same RR regardless of scaling)

# --- Plot bar chart with CI + text annotations ---
labels = ["US (matched counties)", "Appalachia (ARC)"]
rates = [us_rate, app_rate]
cis = [us_ci, app_ci]

lower_err = [rates[i] - cis[i][0] for i in range(2)]
upper_err = [cis[i][1] - rates[i] for i in range(2)]

x = np.arange(2)

fig, ax = plt.subplots(figsize=(6.8, 4.4), dpi=300)

bars = ax.bar(
    x, rates,
    yerr=[lower_err, upper_err],
    capsize=6,
    edgecolor="black",
    linewidth=0.8
)

ax.set_title("Dermatologist density with 95% confidence intervals")
ax.set_ylabel("Dermatologists per 100,000 population")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, max(rates) + 2.0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Bar labels: rate + CI
bar_texts = [
    f"{rates[0]:.2f}\n(95% CI {cis[0][0]:.2f}–{cis[0][1]:.2f})",
    f"{rates[1]:.2f}\n(95% CI {cis[1][0]:.2f}–{cis[1][1]:.2f})",
]
for i, b in enumerate(bars):
    ax.text(
        b.get_x() + b.get_width()/2,
        b.get_height() + 0.15,
        bar_texts[i],
        ha="center", va="bottom", fontsize=9
    )

# Between-bar annotation: p-value + RR
y_annot = max(rates) + 1.2
ax.plot([x[0], x[0], x[1], x[1]], [y_annot-0.2, y_annot, y_annot, y_annot-0.2], lw=1.0, c="black")

p_str = "<0.001" if p_value < 0.001 else f"{p_value:.3f}"
ax.text(
    0.5, y_annot + 0.05,
    f"RR = {rr:.3f} (95% CI {rr_ci_low:.3f}–{rr_ci_high:.3f});  p = {p_str}",
    ha="center", va="bottom", fontsize=9
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64 (95% CI 0.61–0.68), p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

# Draw bracket
ax.plot(
    [x[0], x[0], x[1], x[1]],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1.5,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64, p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

# Draw bracket
ax.plot(
    [x[0], x[0], x[1], x[1]],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1.5,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64, p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

# Draw bracket
ax.plot(
    [x[0], x[0], x[0.5], x[0.5]],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1.5,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64, p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

# Draw bracket
ax.plot(
    [x[0], x[0], x[1], x[1]],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64, p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

bracket_left  = x[0] + 0.15
bracket_right = x[1] - 0.15

ax.plot(
    [bracket_left, bracket_left, bracket_right, bracket_right],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1.5,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# ---- Plug in your final numbers here (these match your screenshot) ----
labels = ["US (matched counties)", "Appalachia (ARC)"]

means = np.array([7.44, 4.79])
ci_low = np.array([7.35, 4.53])
ci_high = np.array([7.53, 5.06])

# Convert CI to asymmetric error bars for matplotlib: [lower_err, upper_err]
yerr = np.vstack([means - ci_low, ci_high - means])

# ---- Figure ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

x = np.arange(len(labels))
colors = ["tab:blue", "tab:orange"]

bars = ax.bar(
    x, means,
    yerr=yerr,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1
)

# Titles / labels
ax.set_title("Dermatologist Density: Appalachia vs. US", fontsize=18, pad=12)
ax.set_ylabel("Dermatologists per 100,000 population", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# Y-axis starts at 0 (and give a little headroom for text)
ymax = (ci_high.max()) * 1.18
ax.set_ylim(0, ymax)

# ---- Annotations: value + CI above the error bar, not overlapping ----
# Set a consistent offset above the top CI for clean spacing
offset = ymax * 0.03  # tweak if you want more/less separation

for i in range(len(labels)):
    top_ci = ci_high[i]
    ax.text(
        x[i],
        top_ci + offset,
        f"{means[i]:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=12
    )

# Optional: clean look
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# ---- RR and p-value annotation (between bars) ----
rr_text = "RR = 0.64, p < 0.001"

# Horizontal bracket height (just above highest CI)
bracket_y = ci_high.max() * 1.10

bracket_left  = x[0] + 0.25
bracket_right = x[1] - 0.25

ax.plot(
    [bracket_left, bracket_left, bracket_right, bracket_right],
    [bracket_y - 0.05, bracket_y, bracket_y, bracket_y - 0.05],
    lw=1.5,
    color="black"
)

# Add text centered between bars
ax.text(
    x.mean(),
    bracket_y + 0.05,
    rr_text,
    ha="center",
    va="bottom",
    fontsize=13
)

plt.tight_layout()

# ---- Save (edit paths as needed) ----
out_png = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.png"
out_pdf = r"C:\Users\victo\AppalachiaDerm\figures\derm_density_app_vs_us_ci.pdf"
plt.savefig(out_png, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")

plt.show()


In [ ]:
county_full.columns

In [ ]:
import pandas as pd

file_path = r"C:\Users\victo\AppalachiaDerm_results.xlsx"

county_full = pd.read_excel(
    file_path,
    sheet_name="County_Level_Data"
)

county_full.head()

In [ ]:
county_full.columns

In [ ]:
summary = pd.read_excel(
    file_path,
    sheet_name="Summary_Stats"
)

summary

In [ ]:
arc_path = r"C:\All Programs\Med School\Appalachia Project Extended\Appalachian Counties Served By ARC\Appalachian-Counties-Served-by-ARC_2021.xlsx"

arc = pd.read_excel(arc_path)

arc.head()

In [ ]:
arc.columns

In [ ]:
subregion_path = r"C:\All Programs\Med School\Appalachia Project Extended\Appalachian Counties Served By ARC\Derms Per 100000 Residents Appalachia Subregions.xlsx"

subregions = pd.read_excel(subregion_path)

subregions.head()

In [ ]:
subregions.columns

In [ ]:
# Standardize county FIPS formatting
subregions["county_fips"] = (
    subregions["FIPS"]
    .astype(str)
    .str.zfill(5)
)

county_full["county_fips"] = (
    county_full["county_fips"]
    .astype(str)
    .str.zfill(5)
)

# Merge subregions into master dataset
county_full = county_full.merge(
    subregions[["county_fips", "SUBREGION"]],
    on="county_fips",
    how="left"
)

# Verify merge
county_full["SUBREGION"].value_counts(dropna=False)

In [ ]:
# Read every sheet in the workbook
all_sheets = pd.read_excel(
    subregion_path,
    sheet_name=None
)

# Show sheet names
all_sheets.keys()

In [ ]:
# Combine all sheets into one dataframe
subregions = pd.concat(
    all_sheets.values(),
    ignore_index=True
)

# Verify
subregions["SUBREGION"].value_counts()

In [ ]:
# Standardize county FIPS formatting
subregions["county_fips"] = (
    subregions["FIPS"]
    .astype(str)
    .str.zfill(5)
)

county_full["county_fips"] = (
    county_full["county_fips"]
    .astype(str)
    .str.zfill(5)
)

# Merge subregions into master dataset
county_full = county_full.drop(columns=["SUBREGION"], errors="ignore")

county_full = county_full.merge(
    subregions[["county_fips", "SUBREGION"]],
    on="county_fips",
    how="left"
)

# Verify
county_full["SUBREGION"].value_counts(dropna=False)

In [ ]:
subregion_summary = (
    county_full
    .dropna(subset=["SUBREGION"])
    .groupby("SUBREGION")
    .agg(
        dermatologist_count=("dermatologist_count", "sum"),
        population=("population", "sum"),
        county_count=("county_fips", "nunique")
    )
)

subregion_summary["derms_per_100k"] = (
    subregion_summary["dermatologist_count"] /
    subregion_summary["population"]
) * 100_000

subregion_summary = subregion_summary.sort_values("derms_per_100k")

subregion_summary

In [ ]:
import numpy as np

subregion_summary["ci_lower"] = (
    (
        subregion_summary["dermatologist_count"]
        - 1.96 * np.sqrt(subregion_summary["dermatologist_count"])
    )
    / subregion_summary["population"]
) * 100_000

subregion_summary["ci_upper"] = (
    (
        subregion_summary["dermatologist_count"]
        + 1.96 * np.sqrt(subregion_summary["dermatologist_count"])
    )
    / subregion_summary["population"]
) * 100_000

subregion_summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

x = np.arange(len(subregion_summary))

means = subregion_summary["derms_per_100k"].values
lower = subregion_summary["ci_lower"].values
upper = subregion_summary["ci_upper"].values

yerr = np.vstack([means - lower, upper - means])

bars = ax.bar(
    x,
    means,
    yerr=yerr,
    capsize=6,
    edgecolor="black",
    linewidth=1
)

ax.set_xticks(x)
ax.set_xticklabels(
    subregion_summary.index,
    rotation=15
)

ax.set_ylabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_title(
    "Dermatologist Density Across Appalachian Subregions",
    fontsize=16,
    pad=15
)

# US average reference line
us_rate = 7.441591

ax.axhline(
    us_rate,
    linestyle="--",
    linewidth=1.8,
    color="red"
)

ax.text(
    len(x)-0.2,
    us_rate + 0.08,
    f"US average = {us_rate:.2f}",
    color="red",
    ha="right",
    fontsize=11
)

# Labels above bars
for i, v in enumerate(means):
    ax.text(
        i,
        upper[i] + 0.10,
        f"{v:.2f}",
        ha="center",
        fontsize=11
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple"
]

bars = ax.bar(
    x,
    means,
    yerr=yerr,
    capsize=6,
    edgecolor="black",
    linewidth=1,
    color=colors
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# Prepare plotting variables
# =========================================================

x = np.arange(len(subregion_summary))

means = subregion_summary["derms_per_100k"].values
lower = subregion_summary["ci_lower"].values
upper = subregion_summary["ci_upper"].values

# Error bars
yerr = np.vstack([
    means - lower,
    upper - means
])

# US national matched-county average
us_rate = 7.441591

# =========================================================
# Create publication-quality figure
# =========================================================

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

# Custom colors
colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple"
]

bars = ax.bar(
    x,
    means,
    yerr=yerr,
    capsize=6,
    edgecolor="black",
    linewidth=1,
    color=colors
)

# =========================================================
# Axis labels and title
# =========================================================

ax.set_xticks(x)

ax.set_xticklabels(
    subregion_summary.index,
    rotation=15,
    fontsize=12
)

ax.set_ylabel(
    "Dermatologists per 100,000 population",
    fontsize=14
)

ax.set_title(
    "Dermatologist Density Across Appalachian Subregions",
    fontsize=20,
    pad=15
)

# =========================================================
# Add US average reference line
# =========================================================

ax.axhline(
    us_rate,
    linestyle="--",
    linewidth=2,
    color="red"
)

ax.text(
    len(x) - 0.1,
    us_rate + 0.03,
    f"US average = {us_rate:.2f}",
    color="red",
    ha="right",
    fontsize=12
)

# =========================================================
# Add value labels above bars
# =========================================================

for i, v in enumerate(means):

    ax.text(
        i,
        upper[i] + 0.12,
        f"{v:.2f}",
        ha="center",
        fontsize=12
    )

# =========================================================
# Final formatting
# =========================================================

ax.set_ylim(0, 8.2)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# Save figure
# =========================================================

out_png = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\subregional_derm_density.png"
)

out_pdf = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\subregional_derm_density.pdf"
)

plt.savefig(
    out_png,
    bbox_inches="tight",
    dpi=300
)

plt.savefig(
    out_pdf,
    bbox_inches="tight"
)

print("Saved PNG:", out_png)
print("Saved PDF:", out_pdf)

# =========================================================
# Show figure
# =========================================================

plt.show()

In [ ]:
subregion_summary[
    [
        "derms_per_100k",
        "ci_lower",
        "ci_upper"
    ]
]

In [ ]:
for region, row in subregion_summary.iterrows():

    print(
        f"{region}: "
        f"{row['derms_per_100k']:.2f} "
        f"(95% CI {row['ci_lower']:.2f}–{row['ci_upper']:.2f})"
    )

In [ ]:
import numpy as np

# US totals
us_derms = county_full["dermatologist_count"].sum()
us_pop = county_full["population"].sum()

# Appalachia totals
app = county_full[county_full["is_appalachia"]]

app_derms = app["dermatologist_count"].sum()
app_pop = app["population"].sum()

# Rates
us_rate = (us_derms / us_pop) * 100_000
app_rate = (app_derms / app_pop) * 100_000

# Poisson CIs
us_lower = (
    (us_derms - 1.96 * np.sqrt(us_derms))
    / us_pop
) * 100_000

us_upper = (
    (us_derms + 1.96 * np.sqrt(us_derms))
    / us_pop
) * 100_000

app_lower = (
    (app_derms - 1.96 * np.sqrt(app_derms))
    / app_pop
) * 100_000

app_upper = (
    (app_derms + 1.96 * np.sqrt(app_derms))
    / app_pop
) * 100_000

print(
    f"US: {us_rate:.2f} "
    f"(95% CI {us_lower:.2f}–{us_upper:.2f})"
)

print(
    f"Appalachia: {app_rate:.2f} "
    f"(95% CI {app_lower:.2f}–{app_upper:.2f})"
)

In [ ]:
# =========================================================
# Counties with zero dermatologists
# =========================================================

# Appalachia counties
app = county_full[county_full["is_appalachia"]]

# Non-Appalachia counties
non_app = county_full[~county_full["is_appalachia"]]

# Zero-dermatologist counties
app_zero = (app["dermatologist_count"] == 0).sum()
non_app_zero = (non_app["dermatologist_count"] == 0).sum()

# Total counties
app_total = len(app)
non_app_total = len(non_app)

# Percentages
app_pct = (app_zero / app_total) * 100
non_app_pct = (non_app_zero / non_app_total) * 100

# Print results
print(
    f"Appalachia: {app_zero}/{app_total} counties "
    f"({app_pct:.1f}%) had zero dermatologists"
)

print(
    f"Non-Appalachia: {non_app_zero}/{non_app_total} counties "
    f"({non_app_pct:.1f}%) had zero dermatologists"
)

In [ ]:
# =========================================================
# Population living in counties with zero dermatologists
# =========================================================

# Appalachia
app = county_full[county_full["is_appalachia"]]

app_zero_pop = app.loc[
    app["dermatologist_count"] == 0,
    "population"
].sum()

app_total_pop = app["population"].sum()

app_zero_pct = (app_zero_pop / app_total_pop) * 100

# Non-Appalachia
non_app = county_full[~county_full["is_appalachia"]]

non_zero_pop = non_app.loc[
    non_app["dermatologist_count"] == 0,
    "population"
].sum()

non_total_pop = non_app["population"].sum()

non_zero_pct = (non_zero_pop / non_total_pop) * 100

print(
    f"Appalachia: {app_zero_pop:,.0f} residents "
    f"({app_zero_pct:.1f}%) lived in counties with zero dermatologists"
)

print(
    f"Non-Appalachia: {non_zero_pop:,.0f} residents "
    f"({non_zero_pct:.1f}%) lived in counties with zero dermatologists"
)

In [ ]:
# =========================================================
# Load melanoma incidence and mortality datasets
# =========================================================

incidence_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Melanoma Data"
    r"\Melanoma_Incidence_CDC_NIH.csv"
)

mortality_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Melanoma Data"
    r"\Melanoma_Mortality_CDC_NIH.csv"
)

incidence = pd.read_csv(incidence_path)
mortality = pd.read_csv(mortality_path)

print("Incidence shape:", incidence.shape)
print("Mortality shape:", mortality.shape)

In [ ]:
incidence.columns

In [ ]:
mortality.columns

In [ ]:
incidence[
    [
        "FIPS",
        "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"
    ]
].head(15)

In [ ]:
mortality[
    [
        "FIPS",
        "Age-Adjusted Death Rate([rate note]) - deaths per 100,000"
    ]
].head(15)

In [ ]:
# =========================================================
# Clean incidence dataset
# =========================================================

incidence_clean = incidence[
    [
        "FIPS",
        "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"
    ]
].copy()

incidence_clean.columns = [
    "county_fips",
    "melanoma_incidence"
]

# Standardize county FIPS
incidence_clean["county_fips"] = (
    incidence_clean["county_fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

# Numeric conversion
incidence_clean["melanoma_incidence"] = pd.to_numeric(
    incidence_clean["melanoma_incidence"],
    errors="coerce"
)

# Remove invalid national row
incidence_clean = incidence_clean[
    incidence_clean["county_fips"] != "00000"
]

# =========================================================
# Clean mortality dataset
# =========================================================

mortality_clean = mortality[
    [
        "FIPS",
        "Age-Adjusted Death Rate([rate note]) - deaths per 100,000"
    ]
].copy()

mortality_clean.columns = [
    "county_fips",
    "melanoma_mortality"
]

mortality_clean["county_fips"] = (
    mortality_clean["county_fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

mortality_clean["melanoma_mortality"] = pd.to_numeric(
    mortality_clean["melanoma_mortality"],
    errors="coerce"
)

mortality_clean = mortality_clean[
    mortality_clean["county_fips"] != "00000"
]

# =========================================================
# Verify cleaned datasets
# =========================================================

print("Incidence rows:", len(incidence_clean))
print("Mortality rows:", len(mortality_clean))

print(
    "Missing incidence:",
    incidence_clean["melanoma_incidence"].isna().sum()
)

print(
    "Missing mortality:",
    mortality_clean["melanoma_mortality"].isna().sum()
)

incidence_clean.head()

In [ ]:
# =========================================================
# Clean incidence dataset — safer FIPS handling
# =========================================================

inc_rate_col = "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"

incidence_clean = incidence[["FIPS", inc_rate_col]].copy()
incidence_clean.columns = ["county_fips", "melanoma_incidence"]

# Convert FIPS safely
incidence_clean["county_fips"] = pd.to_numeric(
    incidence_clean["county_fips"],
    errors="coerce"
)

# Drop missing FIPS and national row FIPS = 0
incidence_clean = incidence_clean.dropna(subset=["county_fips"])
incidence_clean = incidence_clean[incidence_clean["county_fips"] != 0]

# Format FIPS as 5-digit string
incidence_clean["county_fips"] = (
    incidence_clean["county_fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

# Convert incidence to numeric
incidence_clean["melanoma_incidence"] = pd.to_numeric(
    incidence_clean["melanoma_incidence"],
    errors="coerce"
)

# =========================================================
# Clean mortality dataset — safer FIPS handling
# =========================================================

mort_rate_col = "Age-Adjusted Death Rate([rate note]) - deaths per 100,000"

mortality_clean = mortality[["FIPS", mort_rate_col]].copy()
mortality_clean.columns = ["county_fips", "melanoma_mortality"]

mortality_clean["county_fips"] = pd.to_numeric(
    mortality_clean["county_fips"],
    errors="coerce"
)

mortality_clean = mortality_clean.dropna(subset=["county_fips"])
mortality_clean = mortality_clean[mortality_clean["county_fips"] != 0]

mortality_clean["county_fips"] = (
    mortality_clean["county_fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

mortality_clean["melanoma_mortality"] = pd.to_numeric(
    mortality_clean["melanoma_mortality"],
    errors="coerce"
)

# =========================================================
# Verify
# =========================================================

print("Incidence rows:", len(incidence_clean))
print("Mortality rows:", len(mortality_clean))

print("Missing incidence:", incidence_clean["melanoma_incidence"].isna().sum())
print("Missing mortality:", mortality_clean["melanoma_mortality"].isna().sum())

incidence_clean.head(), mortality_clean.head()

In [ ]:
# Avoid duplicate melanoma columns if rerunning
county_full = county_full.drop(
    columns=[
        "melanoma_incidence",
        "melanoma_mortality"
    ],
    errors="ignore"
)

county_full["county_fips"] = county_full["county_fips"].astype(str).str.zfill(5)

county_full = county_full.merge(
    incidence_clean,
    on="county_fips",
    how="left"
)

county_full = county_full.merge(
    mortality_clean,
    on="county_fips",
    how="left"
)

print("Rows:", len(county_full))
print("Missing incidence %:", county_full["melanoma_incidence"].isna().mean())
print("Missing mortality %:", county_full["melanoma_mortality"].isna().mean())

county_full[
    [
        "county_fips",
        "dermatologist_count",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION",
        "melanoma_incidence",
        "melanoma_mortality"
    ]
].head()

In [ ]:
print("Incidence coverage %:", county_full["melanoma_incidence"].notna().mean())
print("Mortality coverage %:", county_full["melanoma_mortality"].notna().mean())

print("Appalachia incidence coverage %:",
      county_full.loc[county_full["is_appalachia"], "melanoma_incidence"].notna().mean())

print("Appalachia mortality coverage %:",
      county_full.loc[county_full["is_appalachia"], "melanoma_mortality"].notna().mean())

In [ ]:
incidence_clean["melanoma_incidence"].isna().sum()
mortality_clean["melanoma_mortality"].isna().sum()

In [ ]:
# =========================================================
# Appalachia vs US melanoma incidence
# =========================================================

app = county_full[county_full["is_appalachia"]]
non_app = county_full[~county_full["is_appalachia"]]

print(
    "US incidence mean:",
    round(county_full["melanoma_incidence"].mean(), 2)
)

print(
    "Appalachia incidence mean:",
    round(app["melanoma_incidence"].mean(), 2)
)

print(
    "Non-Appalachia incidence mean:",
    round(non_app["melanoma_incidence"].mean(), 2)
)

print()

print(
    "Appalachia counties with incidence data:",
    app["melanoma_incidence"].notna().sum()
)

print(
    "Non-Appalachia counties with incidence data:",
    non_app["melanoma_incidence"].notna().sum()
)

In [ ]:
# =========================================================
# Melanoma incidence by Appalachian subregion
# =========================================================

subregion_incidence = (
    county_full
    .dropna(subset=["SUBREGION"])
    .groupby("SUBREGION")
    .agg(
        mean_incidence=("melanoma_incidence", "mean"),
        median_incidence=("melanoma_incidence", "median"),
        counties_with_data=("melanoma_incidence", lambda x: x.notna().sum())
    )
    .sort_values("mean_incidence")
)

subregion_incidence

In [ ]:
from scipy.stats import pearsonr

# =========================================================
# Correlation between dermatologist density and incidence
# =========================================================

corr_df = county_full.dropna(
    subset=[
        "derms_per_100k",
        "melanoma_incidence"
    ]
)

r, p = pearsonr(
    corr_df["derms_per_100k"],
    corr_df["melanoma_incidence"]
)

print(f"Pearson r = {r:.3f}")
print(f"p-value = {p:.5f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# Scatterplot: dermatologist density vs melanoma incidence
# =========================================================

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

ax.scatter(
    corr_df["derms_per_100k"],
    corr_df["melanoma_incidence"],
    alpha=0.6
)

# Regression line
z = np.polyfit(
    corr_df["derms_per_100k"],
    corr_df["melanoma_incidence"],
    1
)

pfit = np.poly1d(z)

xvals = np.linspace(
    corr_df["derms_per_100k"].min(),
    corr_df["derms_per_100k"].max(),
    100
)

ax.plot(
    xvals,
    pfit(xvals),
    linewidth=2
)

# Labels
ax.set_xlabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_ylabel(
    "Melanoma incidence per 100,000",
    fontsize=13
)

ax.set_title(
    "Melanoma Incidence vs Dermatologist Density",
    fontsize=18,
    pad=15
)

# Correlation annotation
ax.text(
    0.05,
    0.95,
    f"Pearson r = {r:.2f}\np < 0.001",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=12
)

# Clean formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
# =========================================================
# Count suppressed values ("3 or fewer")
# =========================================================

inc_rate_col = (
    "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"
)

mort_rate_col = (
    "Age-Adjusted Death Rate([rate note]) - deaths per 100,000"
)

# Convert to string for safe searching
inc_strings = incidence[inc_rate_col].astype(str)
mort_strings = mortality[mort_rate_col].astype(str)

# Suppressed counts
inc_suppressed = inc_strings.str.contains(
    "3 or fewer",
    case=False,
    na=False
).sum()

mort_suppressed = mort_strings.str.contains(
    "3 or fewer",
    case=False,
    na=False
).sum()

print("Suppressed incidence counties:", inc_suppressed)
print("Suppressed mortality counties:", mort_suppressed)

print()

print(
    "Incidence suppression rate:",
    round(inc_suppressed / len(incidence) * 100, 1),
    "%"
)

print(
    "Mortality suppression rate:",
    round(mort_suppressed / len(mortality) * 100, 1),
    "%"
)

In [ ]:
inc_rate_col = "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"
mort_rate_col = "Age-Adjusted Death Rate([rate note]) - deaths per 100,000"

# Count original suppressed text values
inc_suppressed = incidence[inc_rate_col].astype(str).str.contains(
    "3 or fewer", case=False, na=False
).sum()

mort_suppressed = mortality[mort_rate_col].astype(str).str.contains(
    "3 or fewer", case=False, na=False
).sum()

print("Suppressed incidence counties:", inc_suppressed)
print("Suppressed mortality counties:", mort_suppressed)

print("Incidence suppression %:", round(inc_suppressed / len(incidence) * 100, 1))
print("Mortality suppression %:", round(mort_suppressed / len(mortality) * 100, 1))

In [ ]:
# =========================================================
# Counties with BOTH incidence and mortality data
# =========================================================

mel_compare = county_full.dropna(
    subset=[
        "melanoma_incidence",
        "melanoma_mortality"
    ]
)

print("Counties with both datasets:", len(mel_compare))

print()

print(
    "Mean melanoma incidence:",
    round(mel_compare["melanoma_incidence"].mean(), 2)
)

print(
    "Mean melanoma mortality:",
    round(mel_compare["melanoma_mortality"].mean(), 2)
)

In [ ]:
# =========================================================
# Mortality-to-incidence ratio
# =========================================================

mel_compare = mel_compare.copy()

mel_compare["mort_inc_ratio"] = (
    mel_compare["melanoma_mortality"] /
    mel_compare["melanoma_incidence"]
)

print(
    "Overall mortality-to-incidence ratio:",
    round(mel_compare["mort_inc_ratio"].mean(), 3)
)

In [ ]:
# =========================================================
# Mortality-to-incidence ratio by region
# =========================================================

app_mi = mel_compare[
    mel_compare["is_appalachia"]
]["mort_inc_ratio"]

non_app_mi = mel_compare[
    ~mel_compare["is_appalachia"]
]["mort_inc_ratio"]

print(
    "Appalachia mortality/incidence ratio:",
    round(app_mi.mean(), 3)
)

print(
    "Non-Appalachia mortality/incidence ratio:",
    round(non_app_mi.mean(), 3)
)

In [ ]:
# =========================================================
# Appalachia vs US melanoma mortality
# =========================================================

app = county_full[county_full["is_appalachia"]]
non_app = county_full[~county_full["is_appalachia"]]

print(
    "US mortality mean:",
    round(county_full["melanoma_mortality"].mean(), 2)
)

print(
    "Appalachia mortality mean:",
    round(app["melanoma_mortality"].mean(), 2)
)

print(
    "Non-Appalachia mortality mean:",
    round(non_app["melanoma_mortality"].mean(), 2)
)

print()

print(
    "Appalachia counties with mortality data:",
    app["melanoma_mortality"].notna().sum()
)

print(
    "Non-Appalachia counties with mortality data:",
    non_app["melanoma_mortality"].notna().sum()
)

In [ ]:
# =========================================================
# Mortality-to-incidence ratio by Appalachian subregion
# =========================================================

subregion_mi = (
    mel_compare
    .dropna(subset=["SUBREGION"])
    .groupby("SUBREGION")
    .agg(
        mean_mi_ratio=("mort_inc_ratio", "mean"),
        counties=("county_fips", "count")
    )
    .sort_values("mean_mi_ratio")
)

subregion_mi

In [ ]:
mel_compare["SUBREGION"].value_counts(dropna=False)

In [ ]:
from scipy.stats import ttest_ind

# =========================================================
# Compare mortality/incidence ratios
# Appalachia vs Non-Appalachia
# =========================================================

app_mi = mel_compare[
    mel_compare["is_appalachia"]
]["mort_inc_ratio"].dropna()

non_app_mi = mel_compare[
    ~mel_compare["is_appalachia"]
]["mort_inc_ratio"].dropna()

# Welch's t-test
t_stat, p_val = ttest_ind(
    app_mi,
    non_app_mi,
    equal_var=False
)

print("Appalachia mean M/I ratio:", round(app_mi.mean(), 3))
print("Non-Appalachia mean M/I ratio:", round(non_app_mi.mean(), 3))

print()

print("t-statistic:", round(t_stat, 3))
print("p-value:", p_val)

In [ ]:
# =========================================================
# Remove extreme dermatologist-density outliers
# using 99th percentile filtering
# =========================================================

cutoff = corr_df["derms_per_100k"].quantile(0.99)

print("99th percentile cutoff:", round(cutoff, 2))

filtered = corr_df[
    corr_df["derms_per_100k"] <= cutoff
]

print("Original counties:", len(corr_df))
print("Filtered counties:", len(filtered))

In [ ]:
# =========================================================
# Remove extreme dermatologist-density outliers
# using 99th percentile filtering
# =========================================================

cutoff = corr_df["derms_per_100k"].quantile(0.99)

print("99th percentile cutoff:", round(cutoff, 2))

filtered = corr_df[
    corr_df["derms_per_100k"] <= cutoff
]

print("Original counties:", len(corr_df))
print("Filtered counties:", len(filtered))

In [ ]:
from scipy.stats import pearsonr

r_filt, p_filt = pearsonr(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"]
)

print(f"Filtered Pearson r = {r_filt:.3f}")
print(f"Filtered p-value = {p_filt:.5f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

ax.scatter(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"],
    alpha=0.6
)

# Regression line
z = np.polyfit(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"],
    1
)

pfit = np.poly1d(z)

xvals = np.linspace(
    filtered["derms_per_100k"].min(),
    filtered["derms_per_100k"].max(),
    100
)

ax.plot(
    xvals,
    pfit(xvals),
    linewidth=2
)

# Labels
ax.set_xlabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_ylabel(
    "Melanoma incidence per 100,000",
    fontsize=13
)

ax.set_title(
    "Melanoma Incidence vs Dermatologist Density\n(99th percentile filtered)",
    fontsize=18,
    pad=15
)

# Annotation
ax.text(
    0.05,
    0.95,
    f"Pearson r = {r_filt:.2f}\np < 0.001",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=12
)

# Clean formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
# =========================================================
# Melanoma incidence vs dermatologist density
# (99th percentile filtered)
# =========================================================

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# ---------------------------------------------------------
# Prepare dataset
# ---------------------------------------------------------

corr_df = county_full.dropna(
    subset=[
        "derms_per_100k",
        "melanoma_incidence"
    ]
)

# Remove extreme outliers using 99th percentile
cutoff = corr_df["derms_per_100k"].quantile(0.99)

filtered = corr_df[
    corr_df["derms_per_100k"] <= cutoff
]

# ---------------------------------------------------------
# Correlation statistics
# ---------------------------------------------------------

r_filt, p_filt = pearsonr(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"]
)

print(f"Filtered Pearson r = {r_filt:.3f}")
print(f"Filtered p-value = {p_filt:.5f}")

# ---------------------------------------------------------
# Generate figure
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

# Scatter points
ax.scatter(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"],
    alpha=0.6
)

# Regression line
z = np.polyfit(
    filtered["derms_per_100k"],
    filtered["melanoma_incidence"],
    1
)

pfit = np.poly1d(z)

xvals = np.linspace(
    filtered["derms_per_100k"].min(),
    filtered["derms_per_100k"].max(),
    100
)

ax.plot(
    xvals,
    pfit(xvals),
    linewidth=2
)

# ---------------------------------------------------------
# Labels and formatting
# ---------------------------------------------------------

ax.set_xlabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_ylabel(
    "Melanoma incidence per 100,000",
    fontsize=13
)

ax.set_title(
    "Melanoma Incidence vs Dermatologist Density\n"
    "(99th percentile filtered)",
    fontsize=18,
    pad=15
)

# Correlation annotation
ax.text(
    0.05,
    0.95,
    f"Pearson r = {r_filt:.2f}\np < 0.001",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=12
)

# Clean publication formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ---------------------------------------------------------
# Save figure
# ---------------------------------------------------------

png_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\melanoma_incidence_vs_derm_density_filtered.png"
)

pdf_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\melanoma_incidence_vs_derm_density_filtered.pdf"
)

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

plt.show()

In [ ]:
county_full.columns

In [ ]:
# =========================================================
# Create final manuscript dataset
# =========================================================

master_dataset = county_full[
    [
        "county_fips",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION",
        "melanoma_incidence",
        "melanoma_mortality"
    ]
].copy()

master_dataset.head()

In [ ]:
# =========================================================
# Save master manuscript dataset
# =========================================================

output_path = (
    r"C:\Users\victo\AppalachiaDerm"
    r"\Appalachia_Master_Analysis_Dataset.xlsx"
)

with pd.ExcelWriter(output_path) as writer:

    master_dataset.to_excel(
        writer,
        sheet_name="County_Level_Data",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary_Stats",
        index=False
    )

    subregion_summary.to_excel(
        writer,
        sheet_name="Subregional_Analysis"
    )

    subregion_incidence.to_excel(
        writer,
        sheet_name="Melanoma_Incidence"
    )

    subregion_mi.to_excel(
        writer,
        sheet_name="Mortality_Incidence_Ratio"
    )

print("Saved:", output_path)

In [ ]:
# =========================================================
# Create heatmap export dataset
# =========================================================

heatmap_export = county_full[
    [
        "county_fips",
        "county_name",
        "state",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION"
    ]
].copy()

# Preview
heatmap_export.head()

In [ ]:
import pandas as pd

# =========================================================
# Reload finalized master dataset
# =========================================================

file_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\Appalachia_Master_Analysis_Dataset.xlsx"
)

county_full = pd.read_excel(
    file_path,
    sheet_name="County_Level_Data"
)

print(county_full.shape)
county_full.head()

In [ ]:
# =========================================================
# Create heatmap export dataset
# =========================================================

heatmap_export = county_full[
    [
        "county_fips",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION"
    ]
].copy()

heatmap_export.head()

In [ ]:
# =========================================================
# Export county heatmap dataset
# =========================================================

heatmap_output = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\County_Heatmap_Export.xlsx"
)

heatmap_export.to_excel(
    heatmap_output,
    index=False
)

print("Saved:", heatmap_output)

In [ ]:
pop.columns

In [ ]:
# =========================================================
# Rebuild county/state lookup from population dataframe
# =========================================================

# Ensure STATEFP is formatted correctly
pop["STATEFP"] = (
    pop["STATEFP"]
    .astype(str)
    .str.zfill(2)
)

# County names
pop["county_name"] = (
    pop["county_name"]
    .astype(str)
    .str.strip()
)

# State names
pop["state"] = (
    pop["state_name"]
    .astype(str)
    .str.strip()
)

# Create county FIPS
pop["county_fips"] = (
    pop["STATEFP"]
    + pop.index.astype(str)
)

# Keep only needed columns
county_lookup = pop[
    [
        "county_fips",
        "county_name",
        "state"
    ]
].copy()

county_lookup = county_lookup.drop_duplicates()

county_lookup.head()

In [ ]:
import pandas as pd

# =========================================================
# Reload population dataset
# =========================================================

pop_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\NPI registry Data\Large Data File"
    r"\US CENSUS 2020-2024 County Population"
    r"\co-est2024-pop.xlsx"
)

pop = pd.read_excel(pop_path)

print(pop.shape)
pop.head()

In [ ]:
pop.columns

In [ ]:
# =========================================================
# Reload finalized master dataset
# =========================================================

master_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\Appalachia_Master_Analysis_Dataset.xlsx"
)

county_full = pd.read_excel(
    master_path,
    sheet_name="County_Level_Data"
)

county_full.head()

In [ ]:
# =========================================================
# Reload ARC county file
# =========================================================

arc_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Appalachian Counties Served By ARC"
    r"\Appalachian-Counties-Served-by-ARC_2021.xlsx"
)

arc = pd.read_excel(arc_path)

arc.head()

In [ ]:
# =========================================================
# Create county/state lookup table
# =========================================================

arc["county_fips"] = (
    arc["FIPS"]
    .astype(str)
    .str.zfill(5)
)

arc["county_name"] = arc["COUNTY"].astype(str).str.strip()
arc["state"] = arc["STATE"].astype(str).str.strip()

county_lookup = arc[
    [
        "county_fips",
        "county_name",
        "state"
    ]
].drop_duplicates()

county_lookup.head()

In [ ]:
# =========================================================
# Merge county/state names
# =========================================================

county_full["county_fips"] = (
    county_full["county_fips"]
    .astype(str)
    .str.zfill(5)
)

county_full = county_full.merge(
    county_lookup,
    on="county_fips",
    how="left"
)

county_full.head()

In [ ]:
# =========================================================
# Create final heatmap export dataset
# =========================================================

heatmap_export = county_full[
    [
        "county_fips",
        "county_name",
        "state",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION"
    ]
].copy()

heatmap_export.head()

In [ ]:
# =========================================================
# Export Excel heatmap file
# =========================================================

heatmap_output = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\County_Heatmap_Export.xlsx"
)

heatmap_export.to_excel(
    heatmap_output,
    index=False
)

print("Saved:", heatmap_output)

In [ ]:
# =========================================================
# Build nationwide county lookup from melanoma dataset
# =========================================================

county_lookup = incidence[
    [
        "FIPS",
        "County"
    ]
].copy()

# Format FIPS
county_lookup["county_fips"] = (
    county_lookup["FIPS"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

# Clean county names
county_lookup["county_name"] = (
    county_lookup["County"]
    .astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

county_lookup = county_lookup[
    [
        "county_fips",
        "county_name"
    ]
].drop_duplicates()

county_lookup.head()

In [ ]:
import pandas as pd

# =========================================================
# Reload melanoma incidence dataset
# =========================================================

incidence_path = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Melanoma Data"
    r"\Melanoma_Incidence_CDC_NIH.csv"
)

incidence = pd.read_csv(incidence_path)

print(incidence.shape)
incidence.head()

In [ ]:
# =========================================================
# Build nationwide county lookup from melanoma dataset
# =========================================================

county_lookup = incidence[
    [
        "FIPS",
        "County"
    ]
].copy()

# Format FIPS
county_lookup["county_fips"] = (
    county_lookup["FIPS"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

# Clean county names
county_lookup["county_name"] = (
    county_lookup["County"]
    .astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

county_lookup = county_lookup[
    [
        "county_fips",
        "county_name"
    ]
].drop_duplicates()

county_lookup.head()

In [ ]:
# =========================================================
# Remove CDC footnote annotations from county names
# =========================================================

county_lookup["county_name"] = (
    county_lookup["county_name"]
    .str.replace(r"\(\d+\)", "", regex=True)
    .str.strip()
)

county_lookup.head()

In [ ]:
# =========================================================
# Merge county names into county_full
# =========================================================

county_full["county_fips"] = (
    county_full["county_fips"]
    .astype(str)
    .str.zfill(5)
)

county_full = county_full.drop(
    columns=["county_name"],
    errors="ignore"
)

county_full = county_full.merge(
    county_lookup,
    on="county_fips",
    how="left"
)

county_full.head()

In [ ]:
# =========================================================
# Create and export heatmap dataset
# =========================================================

heatmap_export = county_full[
    [
        "county_fips",
        "county_name",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION"
    ]
].copy()

heatmap_output = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\County_Heatmap_Export.xlsx"
)

heatmap_export.to_excel(
    heatmap_output,
    index=False
)

print("Saved:", heatmap_output)

In [ ]:
# =========================================================
# Split county and state into separate columns
# =========================================================

heatmap_export[
    ["county", "state"]
] = heatmap_export["county_name"].str.split(
    ",",
    n=1,
    expand=True
)

# Clean whitespace
heatmap_export["county"] = (
    heatmap_export["county"]
    .str.strip()
)

heatmap_export["state"] = (
    heatmap_export["state"]
    .str.strip()
)

heatmap_export.head()

In [ ]:
heatmap_export = heatmap_export[
    [
        "county_fips",
        "county",
        "state",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "is_appalachia",
        "SUBREGION"
    ]
]

heatmap_export.head()

In [ ]:
# =========================================================
# Export cleaned Excel heatmap file
# =========================================================

heatmap_output = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\County_Heatmap_Export.xlsx"
)

heatmap_export.to_excel(
    heatmap_output,
    index=False
)

print("Saved:", heatmap_output)

In [ ]:
# =========================================================
# Create capped heatmap variable
# =========================================================

# 99th percentile cutoff
cutoff = heatmap_export["derms_per_100k"].quantile(0.99)

print("99th percentile cutoff:", round(cutoff, 2))

# Create capped variable
heatmap_export["derms_per_100k_capped"] = (
    heatmap_export["derms_per_100k"]
    .clip(upper=cutoff)
)

heatmap_export.head()

In [ ]:
heatmap_export["density_bin"] = pd.cut(
    heatmap_export["derms_per_100k"],
    bins=[0, 1, 3, 5, 10, cutoff],
    labels=[
        "0–1",
        "1–3",
        "3–5",
        "5–10",
        f"10–{round(cutoff,1)}"
    ],
    include_lowest=True
)

In [ ]:
import pandas as pd

# =========================================================
# Create 99th percentile capped variable + outlier flag
# =========================================================

# Make sure derm density is numeric
heatmap_export["derms_per_100k"] = pd.to_numeric(
    heatmap_export["derms_per_100k"],
    errors="coerce"
)

# 99th percentile cutoff
cutoff_99 = heatmap_export["derms_per_100k"].quantile(0.99)

print("99th percentile cutoff:", round(cutoff_99, 2))

# Flag outliers at/above 99th percentile
heatmap_export["density_outlier_99th"] = (
    heatmap_export["derms_per_100k"] >= cutoff_99
)

# Capped variable for Excel heatmap visualization
heatmap_export["derms_per_100k_capped_99th"] = (
    heatmap_export["derms_per_100k"].clip(upper=cutoff_99)
)

# Optional: cleaner rounded columns for Excel display
heatmap_export["derms_per_100k"] = heatmap_export["derms_per_100k"].round(2)
heatmap_export["derms_per_100k_capped_99th"] = heatmap_export["derms_per_100k_capped_99th"].round(2)
heatmap_export["dermatologist_count"] = heatmap_export["dermatologist_count"].round(3)

# =========================================================
# Reorder columns for Excel
# =========================================================

heatmap_export = heatmap_export[
    [
        "county_fips",
        "county",
        "state",
        "dermatologist_count",
        "population",
        "derms_per_100k",
        "derms_per_100k_capped_99th",
        "density_outlier_99th",
        "is_appalachia",
        "SUBREGION"
    ]
]

heatmap_export.head()

In [ ]:
# =========================================================
# Export updated heatmap file
# =========================================================

heatmap_output = (
    r"C:\All Programs\Med School\Appalachia Project Extended"
    r"\Final Results Folder"
    r"\County_Heatmap_Export.xlsx"
)

heatmap_export.to_excel(
    heatmap_output,
    index=False
)

print("Saved:", heatmap_output)

In [ ]:
# =========================================================
# Correlation:
# Dermatologist density vs melanoma mortality
# =========================================================

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# ---------------------------------------------------------
# Build correlation dataframe
# ---------------------------------------------------------

mort_corr = county_full.dropna(
    subset=[
        "derms_per_100k",
        "melanoma_mortality"
    ]
).copy()

print("Counties included:", len(mort_corr))

# ---------------------------------------------------------
# Optional:
# Remove extreme derm density outliers (99th percentile)
# ---------------------------------------------------------

cutoff = mort_corr["derms_per_100k"].quantile(0.99)

mort_corr = mort_corr[
    mort_corr["derms_per_100k"] <= cutoff
]

print("99th percentile cutoff:", round(cutoff, 2))
print("Counties after filtering:", len(mort_corr))

# ---------------------------------------------------------
# Pearson correlation
# ---------------------------------------------------------

r_mort, p_mort = pearsonr(
    mort_corr["derms_per_100k"],
    mort_corr["melanoma_mortality"]
)

print(f"Pearson r = {r_mort:.3f}")
print(f"p-value = {p_mort:.5f}")

# ---------------------------------------------------------
# Generate figure
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

# Scatter plot
ax.scatter(
    mort_corr["derms_per_100k"],
    mort_corr["melanoma_mortality"],
    alpha=0.6
)

# Regression line
z = np.polyfit(
    mort_corr["derms_per_100k"],
    mort_corr["melanoma_mortality"],
    1
)

pfit = np.poly1d(z)

xvals = np.linspace(
    mort_corr["derms_per_100k"].min(),
    mort_corr["derms_per_100k"].max(),
    100
)

ax.plot(
    xvals,
    pfit(xvals),
    linewidth=2
)

# Labels
ax.set_xlabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_ylabel(
    "Melanoma mortality per 100,000",
    fontsize=13
)

ax.set_title(
    "Melanoma Mortality vs Dermatologist Density\n"
    "(99th percentile filtered)",
    fontsize=18,
    pad=15
)

# Annotation
ax.text(
    0.05,
    0.95,
    f"Pearson r = {r_mort:.2f}\n"
    f"p = {p_mort:.3f}",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=12
)

# Publication formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ---------------------------------------------------------
# Save figure
# ---------------------------------------------------------

png_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\melanoma_mortality_vs_derm_density.png"
)

pdf_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\melanoma_mortality_vs_derm_density.pdf"
)

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

plt.show()

In [ ]:
# =========================================================
# Appalachia vs Non-Appalachia dermatologist density
# =========================================================

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ttest_ind, sem

# ---------------------------------------------------------
# Create groups
# ---------------------------------------------------------

app = county_full[
    county_full["is_appalachia"]
]["derms_per_100k"].dropna()

non_app = county_full[
    ~county_full["is_appalachia"]
]["derms_per_100k"].dropna()

# ---------------------------------------------------------
# Means and confidence intervals
# ---------------------------------------------------------

app_mean = app.mean()
non_app_mean = non_app.mean()

app_ci = 1.96 * sem(app)
non_app_ci = 1.96 * sem(non_app)

# ---------------------------------------------------------
# Statistical test
# ---------------------------------------------------------

t_stat, p_val = ttest_ind(
    app,
    non_app,
    equal_var=False
)

# Rate ratio
rr = app_mean / non_app_mean

# ---------------------------------------------------------
# Create figure
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

groups = ["Appalachia", "Non-Appalachia"]
means = [app_mean, non_app_mean]
cis = [app_ci, non_app_ci]

bars = ax.bar(
    groups,
    means,
    yerr=cis,
    capsize=8
)

# Color bars
bars[0].set_color("royalblue")
bars[1].set_color("darkorange")

# ---------------------------------------------------------
# Labels and title
# ---------------------------------------------------------

ax.set_ylabel(
    "Dermatologists per 100,000 population",
    fontsize=13
)

ax.set_title(
    "Dermatologist Density:\nAppalachia vs Non-Appalachia",
    fontsize=18,
    pad=20
)

# Start y-axis at zero
ax.set_ylim(bottom=0)

# ---------------------------------------------------------
# Statistical annotation
# ---------------------------------------------------------

y_max = max(means[0] + cis[0], means[1] + cis[1])

# Bracket
ax.plot(
    [0, 0, 1, 1],
    [y_max + 0.3, y_max + 0.7, y_max + 0.7, y_max + 0.3],
    linewidth=1.5
)

# Text annotation
ax.text(
    0.5,
    y_max + 1.0,
    f"RR = {rr:.2f}\np < 0.001",
    ha="center",
    fontsize=11
)

# ---------------------------------------------------------
# Clean formatting
# ---------------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ---------------------------------------------------------
# Save figure
# ---------------------------------------------------------

png_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\appalachia_vs_nonappalachia_density.png"
)

pdf_path = (
    r"C:\Users\victo\AppalachiaDerm\figures"
    r"\appalachia_vs_nonappalachia_density.pdf"
)

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

print()
print("Appalachia mean:", round(app_mean, 2))
print("Non-Appalachia mean:", round(non_app_mean, 2))
print("Rate Ratio:", round(rr, 2))
print("p-value:", p_val)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Groups
app = county_full[county_full["is_appalachia"]].copy()
non_app = county_full[~county_full["is_appalachia"]].copy()

# Population-weighted rates
app_derms = app["dermatologist_count"].sum()
app_pop = app["population"].sum()

non_app_derms = non_app["dermatologist_count"].sum()
non_app_pop = non_app["population"].sum()

app_rate = app_derms / app_pop * 100_000
non_app_rate = non_app_derms / non_app_pop * 100_000

# Poisson CIs
app_ci_low = (app_derms - 1.96 * np.sqrt(app_derms)) / app_pop * 100_000
app_ci_high = (app_derms + 1.96 * np.sqrt(app_derms)) / app_pop * 100_000

non_app_ci_low = (non_app_derms - 1.96 * np.sqrt(non_app_derms)) / non_app_pop * 100_000
non_app_ci_high = (non_app_derms + 1.96 * np.sqrt(non_app_derms)) / non_app_pop * 100_000

# Rate ratio and p-value
rr = app_rate / non_app_rate
se_log_rr = np.sqrt(1/app_derms + 1/non_app_derms)

rr_ci_low = np.exp(np.log(rr) - 1.96 * se_log_rr)
rr_ci_high = np.exp(np.log(rr) + 1.96 * se_log_rr)

z = np.log(rr) / se_log_rr
p_value = 2 * (1 - norm.cdf(abs(z)))

# Plot
labels = ["Appalachia", "Non-Appalachia"]
rates = np.array([app_rate, non_app_rate])
ci_low = np.array([app_ci_low, non_app_ci_low])
ci_high = np.array([app_ci_high, non_app_ci_high])
yerr = np.vstack([rates - ci_low, ci_high - rates])

fig, ax = plt.subplots(figsize=(7, 6), dpi=300)

bars = ax.bar(
    labels,
    rates,
    yerr=yerr,
    capsize=7,
    color=["tab:blue", "tab:orange"],
    edgecolor="black",
    linewidth=1
)

ax.set_title("Dermatologist Density: Appalachia vs Non-Appalachia", fontsize=18, pad=18)
ax.set_ylabel("Dermatologists per 100,000", fontsize=13)
ax.set_ylim(0, max(ci_high) * 1.25)

# Labels above bars
for i, v in enumerate(rates):
    ax.text(
        i,
        ci_high[i] + 0.15,
        f"{v:.2f}\n(95% CI {ci_low[i]:.2f}–{ci_high[i]:.2f})",
        ha="center",
        va="bottom",
        fontsize=11
    )

# Bracket annotation
y_bracket = max(ci_high) * 1.12
ax.plot([0, 0, 1, 1], [y_bracket - 0.08, y_bracket, y_bracket, y_bracket - 0.08],
        color="black", linewidth=1.3)

p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"

ax.text(
    0.5,
    y_bracket + 0.08,
    f"RR = {rr:.2f} (95% CI {rr_ci_low:.2f}–{rr_ci_high:.2f}), {p_text}",
    ha="center",
    va="bottom",
    fontsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

png_path = r"C:\Users\victo\AppalachiaDerm\figures\appalachia_vs_nonappalachia_density_weighted.png"
pdf_path = r"C:\Users\victo\AppalachiaDerm\figures\appalachia_vs_nonappalachia_density_weighted.pdf"

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

print("Appalachia rate:", app_rate)
print("Non-Appalachia rate:", non_app_rate)
print("RR:", rr)
print("p-value:", p_value)
print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

plt.show()